# Gurobi: distance minimization with logical-quotient constraints

**Question.** How can an integer-programming solver find a minimum-weight
logical operator without mistaking a stabilizer for a logical operator?

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu,
Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.
The formulation uses the CSS logical quotient in Section II.C
(`sec:logical_ops_css_code`). It is an implementation method, not an additional
claim attributed to the paper. Equation numbers below are local to this notebook.

Install the optional dependency from the repository root with
`python -m pip install -e '.[gurobi]'`. Select the installed environment and run
all cells in order. A license valid for the model size is required. If Gurobi is
unavailable, this notebook explicitly skips solver results; it does not replace
them with enumeration results. The dedicated Gurobi CI job requires it to work.

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Formulate the Z-sector problem

For a binary CSS code, a Z logical operator must commute with all X checks
but must not lie in the span of the Z stabilizers. Let the rows of $L_X$ be
logical X representatives returned by `code.logical_operators()`. Then

$$
z\in\ker H_X\setminus\operatorname{rowsp}H_Z
\quad\Longleftrightarrow\quad
H_Xz=0\pmod2,\quad L_Xz\ne0\pmod2. \tag{1}
$$

Indeed, the rows of H_X together with L_X span ker H_Z. Their common
orthogonal complement is rowsp H_Z. This excludes every stabilizer, including
nonzero low-weight ones; requiring only z != 0 would be incorrect.

Introduce binary support variables z and binary logical-parity variables s,
with nonnegative integer parity slacks a and b. The MILP is

$$
\begin{aligned}
d_Z=\min\;&\sum_{j=1}^{N}z_j,\\
\text{subject to }&H_Xz=2a,\quad L_Xz=2b+s,\quad\sum_{i=1}^{K}s_i\ge1,\\
&z\in\{0,1\}^N,\quad s\in\{0,1\}^K,\quad a,b\ge0\text{ integer}.
\end{aligned} \tag{2}
$$

Exchanging X and Z gives the other sector. Thus

$$
D=\min(d_X,d_Z),\qquad
\min(\ell_X,\ell_Z)\le D\le\min(u_X,u_Z), \tag{3}
$$

where each sector provides a valid lower bound ell and witness upper bound u.
One solved sector alone does not certify D if the other sector could be smaller.
For K = 0, the library returns `no_logicals` without starting a solver.

In [2]:
import os
from qldpc_generator import hgp
from qldpc_generator.gurobi_distance import GurobiUnavailableError

h = [[1, 1, 0], [0, 1, 1]]
small = hgp(h, h)
GUROBI_AVAILABLE = False
try:
    result = small.distance(method='gurobi', time_limit=10, threads=1)
except GurobiUnavailableError as exc:
    print("Gurobi results skipped:", exc)
    if os.environ.get('QLDPC_REQUIRE_GUROBI') == '1':
        raise
else:
    GUROBI_AVAILABLE = True
    assert result['status'] == 'exact' and result['value'] == 3
    print("Gurobi version:", result['solver_version'])
    show_table(["Sector", "Lower bound", "Upper bound", "Solver status", "Support"], [
        [sector, data['lower_bound'], data['upper_bound'], data['solver_status'], data['witness']]
        for sector, data in result['sectors'].items()
    ])
    for sector, data in result['sectors'].items():
        vector = np.zeros(small.n, dtype=np.uint8)
        vector[data['witness']] = 1
        assert small.is_logical(vector, sector)
        assert int(vector.sum()) == data['upper_bound']

Gurobi version: 14.0.0


| Sector | Lower bound | Upper bound | Solver status | Support |
| --- | --- | --- | --- | --- |
| X | 3 | 3 | OPTIMAL | [4, 5, 6] |
| Z | 3 | 3 | OPTIMAL | [4, 7, 10] |

## 2. Interpret optimality and time limits correctly

The implementation sets both relative and absolute MIP gap tolerances to
zero. Each returned incumbent is checked again using binary arithmetic:
zero syndrome, nonmembership in the stabilizer space, and the stated weight.
The lower bound relies on Gurobi's optimality machinery and numerical
tolerances; it is not a separately checkable proof of infeasibility below
that weight. Small-instance tests compare with independent full enumeration.

`exact` means the combined lower and upper bounds agree. Otherwise the result
is `bounded`, even when a small logical operator has been found. Each sector
records `solver_status`, runtime, bounds, model size, and witness source.
The time limit is **per sector**, excluding model construction; total wall time
can therefore exceed twice the specified limit.

The deliberate zero-second run below stops before accepting an incumbent. A
logical-basis row is still a valid weight-three witness, but only the trivial
lower bound of one is known. It must not become an exact distance claim.

In [3]:
if GUROBI_AVAILABLE:
    limited = small.distance(method='gurobi', time_limit=0)
    assert limited['status'] == 'bounded'
    assert (limited['lower_bound'], limited['upper_bound']) == (1, 3)
    assert 'value' not in limited
    show_table(["Sector", "Solver status", "Bounds", "Witness source", "Solver solutions"], [
        [sector, data['solver_status'], f"{data['lower_bound']} <= d <= {data['upper_bound']}",
         data['witness_source'], data['solution_count']]
        for sector, data in limited['sectors'].items()
    ])

| Sector | Solver status | Bounds | Witness source | Solver solutions |
| --- | --- | --- | --- | --- |
| X | TIME_LIMIT | 1 <= d <= 3 | logical_basis | 0 |
| Z | TIME_LIMIT | 1 <= d <= 3 | logical_basis | 0 |

## 3. Compare against enumeration without assuming a speedup

Both methods receive the same check matrices. MITM uses its original subset
budget and weight limit; Gurobi minimizes over all weights with a five-second
limit per sector. These are different stopping rules, so the table compares
both runtime and what was established. A faster bound is not necessarily
stronger than a slower exact result. Timings depend on hardware and solver
version, and are measured again when this notebook runs.

In [4]:
from time import perf_counter
from qldpc_generator.paper_cases import bb_family, cyclic_bp_case

def describe(result):
    if result['status'] == 'exact':
        return f"D = {result['value']} (exact)"
    lo, hi = result.get('lower_bound'), result.get('upper_bound')
    return f"{lo} <= D <= {hi}" if hi is not None else f"D >= {lo}"

if GUROBI_AVAILABLE:
    rows = []
    for name, instance, limit in [('HGP13', small, 3), ('BB72', bb_family(1), 6),
                                   ('BP108', cyclic_bp_case(6).code, 6)]:
        start = perf_counter()
        enumerated = instance.distance(max_weight=limit)
        mitm_seconds = perf_counter()-start
        start = perf_counter()
        optimized = instance.distance(method='gurobi', time_limit=5)
        mip_seconds = perf_counter()-start
        if enumerated['status'] == optimized['status'] == 'exact':
            assert enumerated['value'] == optimized['value']
        else:
            assert optimized['upper_bound'] >= enumerated.get('value', enumerated.get('lower_bound'))
        rows.append([name, describe(enumerated), f"{mitm_seconds:.3f}",
                     describe(optimized), f"{mip_seconds:.3f}"])
    show_table(["Instance", "MITM result", "MITM seconds", "Gurobi result", "Gurobi seconds"], rows)

| Instance | MITM result | MITM seconds | Gurobi result | Gurobi seconds |
| --- | --- | --- | --- | --- |
| HGP13 | D = 3 (exact) | 0.000 | D = 3 (exact) | 0.002 |
| BB72 | D = 6 (exact) | 0.071 | D = 6 (exact) | 0.154 |
| BP108 | D >= 7 | 0.309 | D = 8 (exact) | 0.749 |

## 4. Apply the solver to the paper instances

The report generator attempts every selected instance with the requested
backend, including the large LP cases that MITM leaves unsearched by default.
The paper's quoted distance is never imposed as a constraint or assumed as a
bound. Literature values remain separate from computed results.

```bash
python -m qldpc_generator.verify_paper --distance-method gurobi --time-limit 60 --output reports/gurobi_validation.json
python -m qldpc_generator.verify_paper --distance-method gurobi --cases bp108 bb144 --time-limit 60 --output reports/gurobi_selected.json
```

Notebook 06 also exposes `DISTANCE_METHOD` and `TIME_LIMIT`. A requested
package/license/model-size failure produces an explicit `unavailable` report
entry and a failing report status; no solver fallback occurs. A zero-syndrome
vector alone is not enough to certify a logical operator.

The [recorded Gurobi report](../reports/gurobi_validation.json) uses ten seconds
per sector. Its bounds describe that particular run, not a claim that every
instance is solved exactly. The [default report](../reports/paper_validation.json)
retains MITM results so readers can compare the two approaches.

Implementation: [MILP backend](../qldpc_generator/gurobi_distance.py) and
[independent tests](../tests/test_gurobi_distance.py).
Solver reference: [optimization status codes](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html),
[objective bounds](https://docs.gurobi.com/projects/optimizer/en/current/reference/attributes/model.html#objbound),
and [MIP gap parameters](https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html#mipgap).